[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/18.%20The%20Yard%20Crane%20%28RTG_RMG%29%20Scheduling%20Problem/P18-Tier-9_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 18. The Yard Crane (RTG/RMG) Scheduling Problem
## Tier 9: A QUBO Formulation and Simulated Annealing

### Purpose and scope

This notebook represents a small yard-crane scheduling problem as a Quadratic Unconstrained Binary Optimization (QUBO) problem, then searches for a low-energy binary assignment using a simulated annealing heuristic. The inputs are jobs, cranes, a discrete time horizon, and penalty weights. The outputs are a binary solution, a decoded schedule, constraint diagnostics, and visualizations.

The search procedure is classical: it runs on a conventional CPU and uses random single-bit and multi-bit changes with a Metropolis-style acceptance rule. Multi-bit proposals are quantum-inspired in motivation, but this implementation does not simulate quantum mechanics, use quantum hardware, or establish quantum speedup.

### Modeling assumptions

- Time is represented by integer start-time slots. A job starting at time $t$ occupies its crane over the half-open interval $[t, t+p_j)$, where $p_j$ is its processing time.
- Every job must be assigned exactly once. A crane cannot process overlapping jobs; the example cranes have capacity one.
- An assignment is valid only if its full processing interval fits within both a job time window and the crane's availability window, and its start is no earlier than the job's release time.
- Completion-time cost and weighted deadline-delay cost are included in the energy. Crane travel, job-location-dependent setup time, crane activation costs, and uncertainty are not modeled. Crane position, speed, and type are descriptive input fields, not terms in this QUBO.
- Penalty weights are finite. A low-energy solution is not automatically feasible: feasibility must be checked after decoding.

### Binary decision variables

Let $C$ be the set of cranes, $J$ the set of jobs, and $T=\{0,1,\ldots,H-1\}$ the candidate start times within the horizon. Define

$$
x_{cjt}=\begin{cases}
1, & \text{if crane }c\text{ starts job }j\text{ at time }t,\\
0, & \text{otherwise.}
\end{cases}
$$

Only assignment variables are included. Activation or completion variables would need explicit objective or constraint terms to affect the optimization; unused variables would only enlarge the search space.

For each job $j$, exact assignment is encouraged by the penalty

$$
P_A\left(\sum_{c\in C}\sum_{t\in T}x_{cjt}-1\right)^2.
$$

For each pair of assignments on the same crane whose occupied intervals overlap, the QUBO adds a positive pairwise penalty. Invalid time-window assignments receive a linear penalty. A feasible assignment at time $t$ contributes completion cost $w_C(t+p_j)$ and, if late, deadline cost $w_D\,q_j\max(0,t+p_j-d_j)$.

The full binary objective has the standard QUBO form

$$
\min_{x\in\{0,1\}^{n}} E(x)=x^{\mathsf T}Qx,
$$

where $n=|C||J|H$. The matrix diagonal stores linear terms; symmetric off-diagonal entries encode pairwise terms. For binary variables, $x_i^2=x_i$. The constant $P_A$ from expanding each exact-assignment square is omitted because it is the same for every candidate solution and therefore does not change the minimizer. In code, `build_qubo_matrix` assembles these terms and `_evaluate_qubo` evaluates the energy.

### Search procedure

The solver starts from a random binary vector. At each sweep, it proposes either a one-bit flip or a multi-bit flip. A lower-energy proposal is accepted; a higher-energy proposal may be accepted with probability

$$
\Pr(\text{accept})=\exp\left(-\frac{\Delta E}{\tau+\gamma}\right),
$$

where $\Delta E$ is the proposed energy increase, $\tau$ is a linearly decreasing temperature, and $\gamma$ is a decreasing parameter that increases the acceptance scale during the search. This is a classical stochastic heuristic, not a quantum annealing process. The best solution encountered is retained.

With $n$ binary variables, the search space contains $2^n$ possible vectors. The implementation stores a dense $n\times n$ matrix, requiring $O(n^2)$ memory; it evaluates a proposed vector using dense matrix operations. The heuristic does not guarantee a global optimum, and its result depends on its random trajectory.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np


np.random.seed(42)
random.seed(42)

In [ ]:
@dataclass
class QuantumJob:
    """Job data used by the QUBO scheduling model."""

    id: str
    location: int
    processing_time: int
    release_time: int
    due_date: int
    priority_weight: float
    time_windows: List[Tuple[int, int]]


@dataclass
class QuantumCrane:
    """Crane data; capacity is one in the pairwise-conflict model."""

    id: str
    crane_type: str
    initial_position: int
    speed: float
    capacity: int
    availability_windows: List[Tuple[int, int]]


@dataclass
class QUBOInstance:
    """Scheduling data and its binary-variable index mapping."""

    jobs: List[QuantumJob]
    cranes: List[QuantumCrane]
    time_horizon: int
    penalty_weights: Dict[str, float]
    num_variables: int = 0
    variable_mapping: Dict[str, int] = field(default_factory=dict)
    reverse_mapping: Dict[int, str] = field(default_factory=dict)
    Q_matrix: Optional[np.ndarray] = None

    def __post_init__(self):
        self._create_variable_mappings()

    def _create_variable_mappings(self):
        """Give each crane-job-start choice one binary-vector index."""
        var_index = 0
        for crane in self.cranes:
            for job in self.jobs:
                for start_time in range(self.time_horizon):
                    var_name = f"x_{crane.id}_{job.id}_{start_time}"
                    self.variable_mapping[var_name] = var_index
                    self.reverse_mapping[var_index] = var_name
                    var_index += 1

        self.num_variables = var_index
        self.Q_matrix = np.zeros((var_index, var_index), dtype=float)


class QuantumOptimizer:
    """Build a scheduling QUBO and search it with simulated annealing."""

    def __init__(self, quantum_annealer_type: str = "simulated"):
        if quantum_annealer_type != "simulated":
            raise ValueError("This notebook implements simulated annealing only.")
        self.quantum_annealer_type = quantum_annealer_type
        self.optimization_history = []
        self.search_metrics = {
            "multi_bit_proposals": 0,
            "accepted_moves": 0,
            "proposed_moves": 0,
        }

    def build_qubo_matrix(self, instance: QUBOInstance) -> np.ndarray:
        """Assemble objective and penalty terms in a symmetric Q matrix."""
        Q = np.zeros((instance.num_variables, instance.num_variables), dtype=float)
        self._add_completion_time_objective(Q, instance)
        self._add_job_assignment_constraint(Q, instance)
        self._add_temporal_constraint(Q, instance)
        self._add_capacity_constraint(Q, instance)
        self._add_time_window_constraint(Q, instance)
        self._add_deadline_penalty(Q, instance)
        instance.Q_matrix = Q
        return Q

    def _add_completion_time_objective(self, Q: np.ndarray, instance: QUBOInstance):
        """Charge each selected assignment for its job's completion time."""
        weight = instance.penalty_weights.get("completion_time", 1.0)
        for crane in instance.cranes:
            for job in instance.jobs:
                for start_time in range(instance.time_horizon):
                    index = instance.variable_mapping[
                        f"x_{crane.id}_{job.id}_{start_time}"
                    ]
                    Q[index, index] += weight * (start_time + job.processing_time)

    def _add_job_assignment_constraint(self, Q: np.ndarray, instance: QUBOInstance):
        """Penalize assignments whose count for a job differs from one."""
        penalty = instance.penalty_weights.get("assignment", 100.0)
        for job in instance.jobs:
            job_indices = [
                instance.variable_mapping[f"x_{crane.id}_{job.id}_{start_time}"]
                for crane in instance.cranes
                for start_time in range(instance.time_horizon)
            ]
            for index in job_indices:
                # The diagonal combines the square's linear term with -2P*x.
                Q[index, index] -= penalty
            for position, first_index in enumerate(job_indices):
                for second_index in job_indices[position + 1:]:
                    Q[first_index, second_index] += penalty
                    Q[second_index, first_index] += penalty

    def _add_temporal_constraint(self, Q: np.ndarray, instance: QUBOInstance):
        """Penalize overlapping jobs assigned to the same crane."""
        penalty = instance.penalty_weights.get("temporal", 50.0)
        for crane in instance.cranes:
            for first_position, first_job in enumerate(instance.jobs):
                for second_job in instance.jobs[first_position + 1:]:
                    for first_start in range(instance.time_horizon):
                        first_end = first_start + first_job.processing_time
                        for second_start in range(instance.time_horizon):
                            second_end = second_start + second_job.processing_time
                            overlaps = not (
                                first_end <= second_start or second_end <= first_start
                            )
                            if overlaps:
                                first_index = instance.variable_mapping[
                                    f"x_{crane.id}_{first_job.id}_{first_start}"
                                ]
                                second_index = instance.variable_mapping[
                                    f"x_{crane.id}_{second_job.id}_{second_start}"
                                ]
                                Q[first_index, second_index] += penalty
                                Q[second_index, first_index] += penalty

    def _add_capacity_constraint(self, Q: np.ndarray, instance: QUBOInstance):
        """Add same-start conflict penalties for cranes of capacity one."""
        penalty = instance.penalty_weights.get("capacity", 30.0)
        for crane in instance.cranes:
            if crane.capacity != 1:
                raise ValueError(
                    "The pairwise capacity model requires crane capacity equal to one."
                )
            for start_time in range(instance.time_horizon):
                indices = [
                    instance.variable_mapping[f"x_{crane.id}_{job.id}_{start_time}"]
                    for job in instance.jobs
                ]
                for position, first_index in enumerate(indices):
                    for second_index in indices[position + 1:]:
                        Q[first_index, second_index] += penalty
                        Q[second_index, first_index] += penalty

    @staticmethod
    def _fits_window(start_time: int, end_time: int, windows: List[Tuple[int, int]]) -> bool:
        """Return whether the complete half-open job interval fits a window."""
        return any(
            start_time >= window_start and end_time <= window_end
            for window_start, window_end in windows
        )

    def _assignment_is_time_feasible(
        self, job: QuantumJob, crane: QuantumCrane, start_time: int
    ) -> bool:
        end_time = start_time + job.processing_time
        return (
            start_time >= job.release_time
            and self._fits_window(start_time, end_time, job.time_windows)
            and self._fits_window(start_time, end_time, crane.availability_windows)
        )

    def _add_time_window_constraint(self, Q: np.ndarray, instance: QUBOInstance):
        """Penalize starts violating release, job-window, or crane-window limits."""
        penalty = instance.penalty_weights.get("time_window", 80.0)
        for job in instance.jobs:
            for crane in instance.cranes:
                for start_time in range(instance.time_horizon):
                    if not self._assignment_is_time_feasible(job, crane, start_time):
                        index = instance.variable_mapping[
                            f"x_{crane.id}_{job.id}_{start_time}"
                        ]
                        Q[index, index] += penalty

    def _add_deadline_penalty(self, Q: np.ndarray, instance: QUBOInstance):
        """Charge priority-weighted lateness for each selected assignment."""
        penalty = instance.penalty_weights.get("deadline", 20.0)
        for job in instance.jobs:
            for crane in instance.cranes:
                for start_time in range(instance.time_horizon):
                    completion_time = start_time + job.processing_time
                    delay = max(0, completion_time - job.due_date)
                    if delay:
                        index = instance.variable_mapping[
                            f"x_{crane.id}_{job.id}_{start_time}"
                        ]
                        Q[index, index] += penalty * delay * job.priority_weight

    def solve_quantum_annealing(
        self,
        Q: np.ndarray,
        temperature: float = 1.0,
        num_sweeps: int = 1000,
    ) -> Dict:
        """Run a classical annealing search with occasional multi-bit proposals."""
        if Q.ndim != 2 or Q.shape[0] != Q.shape[1] or Q.shape[0] == 0:
            raise ValueError("Q must be a non-empty square matrix.")
        if num_sweeps < 1 or temperature <= 0:
            raise ValueError("num_sweeps and temperature must be positive.")

        start_clock = time.perf_counter()
        number_of_variables = Q.shape[0]
        current_solution = np.random.randint(0, 2, number_of_variables)
        current_energy = self._evaluate_qubo(current_solution, Q)
        best_solution = current_solution.copy()
        best_energy = current_energy
        energy_history = []
        self.search_metrics = {
            "multi_bit_proposals": 0,
            "accepted_moves": 0,
            "proposed_moves": 0,
        }

        gamma_initial = 1.0
        gamma_final = 0.01
        for sweep in range(num_sweeps):
            progress = sweep / num_sweeps
            current_temperature = temperature * (1.0 - progress)
            gamma = gamma_initial * (gamma_final / gamma_initial) ** progress

            if random.random() < gamma:
                number_of_flips = random.randint(
                    1, min(5, number_of_variables // 10)
                ) if number_of_variables >= 10 else 1
                flip_indices = random.sample(
                    range(number_of_variables), number_of_flips
                )
                self.search_metrics["multi_bit_proposals"] += 1
            else:
                flip_indices = [random.randrange(number_of_variables)]

            candidate = current_solution.copy()
            candidate[flip_indices] = 1 - candidate[flip_indices]
            candidate_energy = self._evaluate_qubo(candidate, Q)
            energy_change = candidate_energy - current_energy
            self.search_metrics["proposed_moves"] += 1

            if energy_change <= 0:
                accept = True
            else:
                acceptance_scale = current_temperature + gamma
                acceptance_probability = np.exp(-energy_change / acceptance_scale)
                accept = random.random() < acceptance_probability

            if accept:
                current_solution = candidate
                current_energy = candidate_energy
                self.search_metrics["accepted_moves"] += 1

            if current_energy < best_energy:
                best_solution = current_solution.copy()
                best_energy = current_energy
            energy_history.append(current_energy)

        elapsed = time.perf_counter() - start_clock
        self.optimization_history = energy_history
        return {
            "solution": best_solution,
            "energy": best_energy,
            "energy_history": energy_history,
            "computation_time": elapsed,
            "search_metrics": self.search_metrics.copy(),
        }

    @staticmethod
    def _evaluate_qubo(solution: np.ndarray, Q: np.ndarray) -> float:
        """Evaluate the binary quadratic energy x-transpose Q x."""
        return float(solution @ Q @ solution)

    def decode_solution(self, solution: np.ndarray, instance: QUBOInstance) -> Dict:
        """Decode selected assignment variables and report objective components."""
        assignments = []
        crane_utilization = defaultdict(list)
        job_assignment_counts = defaultdict(int)
        completions_by_job = defaultdict(list)
        base_completion_cost = 0.0
        weighted_delay = 0.0

        for crane in instance.cranes:
            for job in instance.jobs:
                for start_time in range(instance.time_horizon):
                    index = instance.variable_mapping[
                        f"x_{crane.id}_{job.id}_{start_time}"
                    ]
                    if solution[index] != 1:
                        continue
                    end_time = start_time + job.processing_time
                    assignment = {
                        "crane_id": crane.id,
                        "job_id": job.id,
                        "start_time": start_time,
                        "end_time": end_time,
                        "location": job.location,
                    }
                    assignments.append(assignment)
                    crane_utilization[crane.id].append((start_time, end_time))
                    job_assignment_counts[job.id] += 1
                    completions_by_job[job.id].append(end_time)
                    base_completion_cost += end_time
                    weighted_delay += max(0, end_time - job.due_date) * job.priority_weight

        job_completions = {
            job_id: completion_times[0]
            for job_id, completion_times in completions_by_job.items()
            if len(completion_times) == 1
        }
        deadline_weight = instance.penalty_weights.get("deadline", 20.0)
        return {
            "assignments": assignments,
            "crane_utilization": crane_utilization,
            "job_completions": job_completions,
            "job_assignment_counts": dict(job_assignment_counts),
            "base_completion_cost": base_completion_cost,
            "weighted_delay": weighted_delay,
            "penalized_objective": base_completion_cost + deadline_weight * weighted_delay,
        }

    def check_schedule(self, schedule: Dict, instance: QUBOInstance) -> Dict:
        """Check assignment, time-window, and crane non-overlap feasibility."""
        counts = schedule["job_assignment_counts"]
        exactly_once = all(counts.get(job.id, 0) == 1 for job in instance.jobs)
        time_feasible = True
        for assignment in schedule["assignments"]:
            job = next(job for job in instance.jobs if job.id == assignment["job_id"])
            crane = next(
                crane for crane in instance.cranes
                if crane.id == assignment["crane_id"]
            )
            if not self._assignment_is_time_feasible(
                job, crane, assignment["start_time"]
            ):
                time_feasible = False

        no_overlap = True
        for crane in instance.cranes:
            intervals = sorted(schedule["crane_utilization"].get(crane.id, []))
            for previous, current in zip(intervals, intervals[1:]):
                if previous[1] > current[0]:
                    no_overlap = False

        return {
            "exactly_one_assignment_per_job": exactly_once,
            "time_windows_and_release_times": time_feasible,
            "crane_capacity": no_overlap,
            "feasible": exactly_once and time_feasible and no_overlap,
        }

In [ ]:
def create_quantum_scenario() -> QUBOInstance:
    """Create the job, crane, horizon, and penalty data for the example."""
    jobs = [
        QuantumJob("job_1", 2, 3, 0, 15, 2.0, [(0, 20)]),
        QuantumJob("job_2", 5, 4, 2, 18, 1.5, [(2, 25)]),
        QuantumJob("job_3", 8, 2, 1, 12, 2.5, [(1, 15)]),
        QuantumJob("job_4", 3, 5, 3, 20, 1.0, [(3, 30)]),
        QuantumJob("job_5", 6, 3, 0, 16, 1.8, [(0, 22)]),
    ]

    cranes = [
        QuantumCrane("crane_1", "RTG", 0, 1.0, 1, [(0, 30)]),
        QuantumCrane("crane_2", "RMG", 10, 0.8, 1, [(0, 30)]),
    ]

    penalty_weights = {
        "completion_time": 1.0,
        "assignment": 100.0,
        "temporal": 50.0,
        "capacity": 30.0,
        "time_window": 80.0,
        "deadline": 20.0,
    }

    instance = QUBOInstance(
        jobs=jobs,
        cranes=cranes,
        time_horizon=25,
        penalty_weights=penalty_weights,
    )

    print("=== SCHEDULING QUBO SCENARIO ===")
    print(f"Jobs: {len(jobs)}")
    print(f"Cranes: {len(cranes)}")
    print(f"Time horizon: {instance.time_horizon} start-time slots")
    print(f"Binary assignment variables: {instance.num_variables}")
    print(f"Penalty weights: {penalty_weights}")
    print(
        "Search-space size: 2^{:,} binary vectors (approximately 10^{:.2f})".format(
            instance.num_variables,
            instance.num_variables * np.log10(2),
        )
    )
    return instance


qubo_instance = create_quantum_scenario()

In [ ]:
def run_quantum_optimization(instance: QUBOInstance) -> Dict:
    """Build the QUBO, run the annealing search, and inspect its schedule."""
    print("\n=== BUILDING AND SEARCHING THE QUBO ===")
    optimizer = QuantumOptimizer(quantum_annealer_type="simulated")

    build_start = time.perf_counter()
    Q_matrix = optimizer.build_qubo_matrix(instance)
    build_time = time.perf_counter() - build_start

    print(f"QUBO construction time: {build_time:.3f} seconds")
    print(f"Matrix shape: {Q_matrix.shape}")
    print(f"Nonzero matrix entries: {np.count_nonzero(Q_matrix)}")
    print(f"Matrix density: {np.count_nonzero(Q_matrix) / Q_matrix.size:.6f}")
    print(f"Matrix is symmetric: {np.allclose(Q_matrix, Q_matrix.T)}")
    print(
        "Diagonal entries (linear terms): "
        f"{np.count_nonzero(np.diag(Q_matrix))}"
    )
    off_diagonal = Q_matrix.copy()
    np.fill_diagonal(off_diagonal, 0.0)
    print(f"Nonzero off-diagonal entries: {np.count_nonzero(off_diagonal)}")

    result = optimizer.solve_quantum_annealing(
        Q_matrix,
        temperature=10.0,
        num_sweeps=2000,
    )
    schedule = optimizer.decode_solution(result["solution"], instance)
    feasibility = optimizer.check_schedule(schedule, instance)

    print(f"\nAnnealing search time: {result['computation_time']:.3f} seconds")
    print(f"Best QUBO energy: {result['energy']:.2f}")
    print(f"Decoded assignments: {len(schedule['assignments'])}")
    print(f"Jobs assigned exactly once: {len(schedule['job_completions'])}/{len(instance.jobs)}")
    print(f"Base completion-time cost: {schedule['base_completion_cost']:.2f}")
    print(f"Weighted delay before its QUBO weight: {schedule['weighted_delay']:.2f}")
    print(f"Decoded objective without feasibility penalties: {schedule['penalized_objective']:.2f}")
    print(f"Feasibility checks: {feasibility}")
    print(f"Search diagnostics: {result['search_metrics']}")

    print("\nSelected assignments:")
    for assignment in sorted(
        schedule["assignments"],
        key=lambda item: (item["start_time"], item["crane_id"]),
    ):
        print(
            f"  {assignment['crane_id']} -> {assignment['job_id']} "
            f"(time {assignment['start_time']}–{assignment['end_time']}, "
            f"location {assignment['location']})"
        )

    return {
        "schedule": schedule,
        "quantum_result": result,
        "feasibility": feasibility,
        "optimizer": optimizer,
    }


quantum_results = run_quantum_optimization(qubo_instance)

In [ ]:
def compare_quantum_vs_classical(
    instance: QUBOInstance,
    quantum_results: Dict,
) -> Dict:
    """Summarize the annealing result without asserting quantum advantage."""
    schedule = quantum_results["schedule"]
    result = quantum_results["quantum_result"]
    assigned_count = len(schedule["job_completions"])
    assignment_rate = assigned_count / len(instance.jobs) if instance.jobs else 0.0

    summary = {
        "qubo_energy": result["energy"],
        "decoded_objective": schedule["penalized_objective"],
        "assignment_rate": assignment_rate,
        "feasible": quantum_results["feasibility"]["feasible"],
        "computation_time": result["computation_time"],
    }

    print("\n=== SEARCH RESULT SUMMARY ===")
    print(f"Best penalized QUBO energy: {summary['qubo_energy']:.2f}")
    print(f"Decoded objective, excluding constraint penalties: {summary['decoded_objective']:.2f}")
    print(f"Jobs assigned exactly once: {assigned_count}/{len(instance.jobs)}")
    print(f"Assignment rate: {100 * assignment_rate:.1f}%")
    print(f"Schedule feasible under modeled constraints: {summary['feasible']}")
    print(f"Search computation time: {summary['computation_time']:.3f} seconds")
    print(
        "These observations describe one stochastic heuristic run; they do not "
        "measure quantum speedup or certify global optimality."
    )
    return summary


comparison_results = compare_quantum_vs_classical(qubo_instance, quantum_results)

In [ ]:
def visualize_quantum_results(
    quantum_results: Dict,
    comparison_results: Dict,
    instance: QUBOInstance,
):
    """Plot energy history, assignments, feasibility, and the decoded schedule."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("QUBO Yard-Crane Scheduling: Simulated Annealing", fontsize=15)

    energy_history = quantum_results["quantum_result"]["energy_history"]
    axes[0, 0].plot(energy_history, color="#1f77b4", linewidth=1.2, label="Current energy")
    axes[0, 0].axhline(
        quantum_results["quantum_result"]["energy"],
        color="#d62728",
        linestyle="--",
        label="Best energy encountered",
    )
    axes[0, 0].set_title("Annealing Search History")
    axes[0, 0].set_xlabel("Sweep")
    axes[0, 0].set_ylabel("QUBO energy")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    assignments = quantum_results["schedule"]["assignments"]
    job_names = [job.id for job in instance.jobs]
    job_counts = quantum_results["schedule"]["job_assignment_counts"]
    axes[0, 1].bar(
        job_names,
        [job_counts.get(job_id, 0) for job_id in job_names],
        color="#4c78a8",
    )
    axes[0, 1].axhline(1, color="#d62728", linestyle="--", label="Required count")
    axes[0, 1].set_title("Selected Assignments per Job")
    axes[0, 1].set_xlabel("Job")
    axes[0, 1].set_ylabel("Selected binary assignments")
    axes[0, 1].legend()
    axes[0, 1].grid(True, axis="y", alpha=0.3)

    check_names = [
        "Exactly one\nassignment per job",
        "Time windows and\nrelease times",
        "Crane capacity",
    ]
    check_values = [
        quantum_results["feasibility"]["exactly_one_assignment_per_job"],
        quantum_results["feasibility"]["time_windows_and_release_times"],
        quantum_results["feasibility"]["crane_capacity"],
    ]
    axes[1, 0].bar(
        check_names,
        [int(value) for value in check_values],
        color=["#59a14f" if value else "#e15759" for value in check_values],
    )
    axes[1, 0].set_ylim(0, 1.15)
    axes[1, 0].set_yticks([0, 1], labels=["Not satisfied", "Satisfied"])
    axes[1, 0].set_title("Modeled Feasibility Checks")
    axes[1, 0].grid(True, axis="y", alpha=0.3)

    crane_positions = {crane.id: index for index, crane in enumerate(instance.cranes)}
    crane_colors = {
        crane.id: color
        for crane, color in zip(instance.cranes, ["#4c78a8", "#f28e2b"])
    }
    for assignment in assignments:
        y_position = crane_positions[assignment["crane_id"]]
        duration = assignment["end_time"] - assignment["start_time"]
        axes[1, 1].barh(
            y_position,
            duration,
            left=assignment["start_time"],
            height=0.55,
            color=crane_colors[assignment["crane_id"]],
            label=assignment["crane_id"],
        )
        axes[1, 1].text(
            assignment["start_time"] + duration / 2,
            y_position,
            assignment["job_id"],
            ha="center",
            va="center",
            fontsize=8,
        )
    axes[1, 1].set_yticks(
        range(len(instance.cranes)),
        labels=[f"{crane.id} ({crane.crane_type})" for crane in instance.cranes],
    )
    axes[1, 1].set_xlim(0, instance.time_horizon)
    axes[1, 1].set_xlabel("Time (discrete model units)")
    axes[1, 1].set_title("Decoded Crane Schedule")
    axes[1, 1].grid(True, axis="x", alpha=0.3)

    handles, labels = axes[1, 1].get_legend_handles_labels()
    unique_labels = dict(zip(labels, handles))
    if unique_labels:
        axes[1, 1].legend(unique_labels.values(), unique_labels.keys(), title="Crane")

    fig.tight_layout()
    plt.show()


visualize_quantum_results(quantum_results, comparison_results, qubo_instance)

In [ ]:
def analyze_quantum_advantages(
    instance: QUBOInstance,
    quantum_results: Dict,
    comparison_results: Dict,
) -> Dict:
    """Explain model scale and heuristic diagnostics without quantum claims."""
    number_of_variables = instance.num_variables
    log10_search_space = number_of_variables * np.log10(2)
    matrix_entries = number_of_variables ** 2
    result = quantum_results["quantum_result"]
    checks = quantum_results["feasibility"]

    analysis = {
        "number_of_variables": number_of_variables,
        "log10_search_space_size": log10_search_space,
        "dense_matrix_entries": matrix_entries,
        "feasible": checks["feasible"],
        "search_diagnostics": result["search_metrics"],
    }

    print("\n=== MODEL-SCALE AND RESULT INTERPRETATION ===")
    print(f"Binary assignment variables: {number_of_variables}")
    print(f"Candidate binary vectors: approximately 10^{log10_search_space:.2f}")
    print(f"Dense Q matrix entries: {matrix_entries:,}")
    print(f"Dense matrix storage grows quadratically with the variable count.")
    print(f"Feasible decoded schedule: {analysis['feasible']}")
    print(f"Search diagnostics: {analysis['search_diagnostics']}")
    print(
        "The number of possible vectors describes the model's search space, "
        "not the number of candidates examined by this heuristic."
    )
    print(
        "The implementation is a classical simulated-annealing heuristic. "
        "Its run time and solution quality depend on the instance, penalty "
        "weights, random seed, temperature schedule, and sweep count."
    )
    print(
        "The schedule omits travel and setup times, and finite penalties do "
        "not guarantee that the lowest-energy vector satisfies every constraint."
    )
    return analysis


quantum_advantages = analyze_quantum_advantages(
    qubo_instance,
    quantum_results,
    comparison_results,
)

## Interpreting the result

The search returns the lowest-energy binary vector encountered during its run. The decoded objective reports completion-time and weighted-delay costs; the QUBO energy also includes assignment, overlap, capacity, and time-window penalties. These quantities are not interchangeable. The assignment-rate and feasibility checks reveal whether the selected vector satisfies the modeled scheduling rules.

A feasible result is a valid schedule under the stated assumptions, not a proof that it is globally optimal. An infeasible result indicates that this run did not find a vector satisfying all modeled constraints; finite penalty weights and stochastic search make that possible. The energy-history plot shows the trajectory of the current search state, while the dashed reference marks the best energy observed over the run.

### Scope of the method

The QUBO representation makes the finite decision space explicit and provides a common energy function for objective terms and constraint penalties. The simulated annealing code is an educational heuristic for searching that space. It does not demonstrate quantum parallelism, quantum tunneling, hardware performance, or a quantum advantage. A genuine quantum-annealing experiment would require suitable hardware or a quantum-annealing simulator and a separately validated encoding and performance study.